<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/stage_06_00_model_training_plan.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06 – Model Training – Objetivo y Modelos**


# **1. Plan de entrenamiento**

En este notebook se establecerá el plan de entrenamiento de modelos predictivos sobre el MNQ, manteniendo un esquema experimental consistente y comparable.

**Targets**

Se evaluarán cuatro variables objetivo:

- `delta_60`
- `delta_90`
- `ret_60`
- `ret_90`

**Window size**

Se utilizarán distintos tamaños de ventana histórica: `[30, 60 , 90, 120, 180]`
Cada ventana representa la cantidad de minutos utilizados como contexto de entrada.

**Features**

Todas las ventanas cuentan con 36 features de predicción:

```python
features_to_windows = [
    'minute_of_day',
    'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed',
    'is_mon', 'is_tue', 'is_wed', 'is_thu','is_fri',
    'close',
    'atr_norm_14', 'atr_norm_14_flag', 'atr_norm_20', 'atr_norm_20_flag',
    'ema_60', 'ema_60_flag',
    'mom_10', 'mom_10_flag', 'mom_5', 'mom_5_flag',
    'roc_20', 'roc_20_flag', 'roc_30', 'roc_30_flag', 'roc_60', 'roc_60_flag',
    'roc60_x_atr20', 'roc60_x_atr20_flag',
    'roc60_x_atr14', 'roc60_x_atr14_flag',
    'roc20_minus_roc60', 'roc20_minus_roc60_flag',
    'mom5_minus_mom10', 'mom5_minus_mom10_flag',
    ]
```

**Enfoques de predicción**

- **`seq2one`**: la secuencia de entrada genera una única predicción.
- **`seq2seq`**: la secuencia de entrada genera una secuencia de salida.

**Objetivo**

Comparar sistemáticamente:
- Puntos vs retornos  
- Sensibilidad al tamaño de ventana  
- Diferencias entre enfoques `many-to-one` y `many-to-many`

# **2. Estructura dimensional del problema de predicción**

## **2.1. Ventana de entrada $X$**

Para ambos enfoques de predicción, las ventanas de entrada $X$ tienen la siguiente dimensión:

$$
L \times N
$$

donde:

- $L$ representa la longitud de la ventana histórica y puede tomar los valores $[30, 60, 90, 120, 180]$.
- $N$ corresponde al número de variables predictoras (features), que en todos los casos es igual a **36**.


## **2.2 Ventana de salida $y$**

### **2.2.1. Enfoque `seq2one`**


En este esquema se entrenarán modelos **seq2one** para predecir un **único valor escalar** asociado a cada ventana histórica.

- **Salida / Target (Y)**

  $$
  Y \in \mathbb{R}^{1}
  $$

  Este valor representa la variable objetivo futura definida en los stages previos.

- **Interpretación**

  El modelo aprende una función del tipo:

  $$
  f: \mathbb{R}^{L \times N} \rightarrow \mathbb{R}
  $$

  Es decir, a partir de una ventana histórica de tamaño $L \times N$, el modelo produce **una única predicción final**.

  Este enfoque es adecuado cuando el objetivo es predecir el retorno o delta acumulado a un horizonte fijo $H$.

### **2.2.2. Enfoque `seq2seq`**


En este esquema se entrenarán modelos **seq2seq** para predecir una **secuencia futura completa** a partir de una ventana histórica intradía.

- **Salida / Target (Y)**

  $$
  Y \in \mathbb{R}^{L \times 1}
  $$

  donde:

  - $L$ es la longitud de la secuencia predicha.
  - $1$ corresponde a una variable objetivo escalar por cada paso temporal.

- **Interpretación**

  El modelo aprende una función del tipo:

  $$
  f: \mathbb{R}^{L \times N} \rightarrow \mathbb{R}^{L \times 1}
  $$

  En este caso, el modelo no predice un único valor final, sino la **trayectoria temporal completa** de $L$ pasos futuros.

  Este enfoque es adecuado cuando se desea modelar la dinámica temporal completa del horizonte futuro, y no únicamente su valor agregado final.



# **3. Alcance y criterios metodológicos del stage_06**

## **3.1. Qué se evalúa en este stage**

En el **stage_06** no se realiza la comparación final entre modelos ni se selecciona el “mejor modelo”.

Este stage garantiza que:

- Cada modelo es entrenado correctamente.
- Se respetan estrictamente los splits temporales definidos.
- La complejidad, arquitectura y regularización están fijadas antes del entrenamiento.
- El proceso es reproducible y comparable entre configuraciones.

La comparación real y la evaluación definitiva se realizan recién en el **stage_07**.

**Nota:** El set de validation puede utilizarse únicamente para control de entrenamiento (por ejemplo, early stopping o verificación de convergencia), pero no para seleccionar modelos.



## **3.2. Qué NO es este problema**


Quedan explícitamente fuera de alcance en este stage:

- Predicción de trayectorias completas multi-step adicionales fuera del esquema formal definido.
- Targets agregados ad-hoc (suma, promedio o retornos acumulados recalculados fuera del target previamente definido).
- Entrenamiento de múltiples regresiones independientes por paso futuro sin estructura temporal compartida.
- Modificaciones del esquema `seq2one` o `seq2seq` fuera de la formulación establecida.

Este stage se limita exclusivamente a entrenar modelos bajo una formulación bien definida y controlada del problema.



# **4.  Protocolo experimental común de entrenamiento**

Se entrenan múltiples modelos bajo reglas estrictamente comunes, garantizando comparabilidad entre configuraciones:

- Mismo dataset.
- Mismo split temporal (train / valid / test).
- Mismo escalamiento.
- Misma definición del target.
- Misma función objetivo.

Ejemplos típicos (alineados al enfoque del libro):

- Naive (baseline)
- Modelos lineales (Ridge, Lasso)
- Árboles y ensembles
- Redes neuronales (MLP, LSTM, etc.)

Cada modelo:

- Se entrena utilizando **exclusivamente el set de entrenamiento**.
- Tiene hiperparámetros y regularización definidos previamente.
- No se realizan ajustes posteriores basados en métricas del set de test.

El objetivo de este protocolo es asegurar que cualquier diferencia observada en desempeño se deba al modelo y no a variaciones en el proceso experimental.


# **5. Modelos a evaluar (comparación escalonada)**


Se define un baseline obligatorio para establecer un piso mínimo de desempeño y detectar rápidamente sobreajuste o complejidad innecesaria.

La evaluación se realiza de manera escalonada, comenzando por modelos simples y aumentando progresivamente la capacidad.

Se evaluarán varios modelos combinando:

- 4 targets (`delta_60`, `delta_90`, `ret_60`, `ret_90`)
- 5 window_size (`[30, 60, 90, 120, 180]`)
- Múltiples familias de modelos (baseline, lineales, recurrentes, convolucionales y atención)


## **5.1. Enfoque `seq2one`**

### **5.1.1. Modelos propuestos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    - Predice el último valor observado de la ventana.
    - Define el piso mínimo de performance.
    - Modelo de control.
    - Notebook: `stage_07_01_naive_seq2one.ipynb`


#### **(B) Modelos clásicos**


2. **Ridge / Lasso**
    - Modelos lineales sobre la ventana aplanada (60x20 → 1200).
    - Referencia interpretable y estable.
    - Notebook: `stage_07_02_ridge_seq2one.ipynb`
    - Notebook: `stage_07_03_lasso_seq2one.ipynb`

3. **MLP (Feedforward)**
    - Entrada: ventana aplanada.
    - Salida: escalar.
    - Primera referencia no lineal.
    - Notebook: `stage_07_04_mlp_seq2one.ipynb`



#### **(C) Modelos temporales (many-to-one)**


4. **LSTM many-to-one**

    - Encoder recurrente.
    - Se utiliza únicamente el último estado oculto.
    - Salida escalar.
    - Notebook: `stage_07_05_lstm_seq2one.ipynb`

5. **GRU many-to-one**

    - Variante más simple y estable que LSTM.
    - Notebook: `stage_07_06_gru_seq2one.ipynb`

#### **(D) Modelo no recurrente robusto**


6. **TCN many-to-one**

    - Convoluciones causales dilatadas.
    - Último timestep → proyección escalar.
    - Muy buena estabilidad intradía.
    - Notebook: `stage_07_07_tcn_seq2one.ipynb`


#### **(E)  Modelo con atención**


7. **Transformer encoder-only (seq2one)**

    - Solo encoder.
    - Pooling o token final → salida escalar.
    - Mayor capacidad, mayor riesgo.
    - Notebook: `stage_07_08_transformer_seq2one.ipynb`

### **5.1.2. Regularización - `seq2one`**

La regularización forma parte del diseño de cada modelo y es clave para controlar
la capacidad y garantizar generalización.  
Cada familia de modelos requiere un esquema distinto.

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    **Regularización:** No aplica.

    - No tiene parámetros entrenables.
    - No puede sobreajustar en sentido de Machine Learning.
    - Rol: define el piso mínimo de performance.
    - Si un modelo entrenable no supera este baseline en validation, se descarta.

#### **(B) Modelos clásicos**


2. **Ridge / Lasso**

    **Riesgo:** Bajo, controlado por diseño.

    **Regularización recomendada:**

    - Penalización L2 (Ridge) o L1 (Lasso) como mecanismo principal.
    - Selección del coeficiente de regularización previa al entrenamiento.
    - No requiere dropout ni early stopping.

    **Rol:** referencia lineal, estable e interpretable.

3. **MLP (Feedforward, seq2one)**

    **Riesgo principal:** Sobreajuste por capacidad del modelo.

    **Regularización recomendada:**

    - Penalización L2 (weight decay) como mecanismo principal.
    - Early stopping monitoreando la pérdida en validation.
    - Arquitectura limitada:
      - pocas capas (1–2),
      - número reducido de neuronas.
    - Dropout leve (opcional, solo si se observa sobreajuste claro).

    **Interpretación:** modelo no lineal flexible que requiere regularización explícita.


#### **(C) Modelos temporales (many-to-one)**


4. **LSTM many-to-one**

    **Riesgo:** Alta capacidad combinada con memoria de largo plazo.

    **Regularización recomendada:**

    - Early stopping obligatorio.
    - Tamaño moderado del hidden state.
    - Número de capas limitado (1–2).
    - Dropout en entradas y salidas (no recurrente).
    - Penalización L2 suave (opcional).

    **Nota:** se utiliza únicamente el último estado oculto para la predicción escalar.


5. **GRU many-to-one**

    **Riesgo:** Menor que LSTM, pero presente.

    **Regularización recomendada:**

    - Early stopping como mecanismo principal.
    - Limitar dimensión del hidden state.
    - Limitar número de capas.
    - Dropout opcional entre capas (no dentro de la recurrencia).

    **Ventaja:** mayor estabilidad y menor necesidad de regularización que LSTM.


#### **(D) Modelo no recurrente robusto**


6. **TCN (Temporal Convolutional Network, seq2one)**

    **Riesgo:** Campo receptivo excesivo o filtros redundantes.

    **Regularización recomendada:**

    - Limitar profundidad del modelo (dilataciones).
    - Limitar número de filtros por capa.
    - Dropout (especialmente efectivo en TCN).
    - Early stopping.
    - Weight normalization (si está disponible).

    **Ventaja clave:** buena generalización intradía con menor complejidad recurrente.


#### **(E) Modelo con atención**


7. **Transformer encoder-only (seq2one)**

    **Riesgo:** Sobreajuste por alta capacidad.

    **Regularización obligatoria:**

    - Dropout en bloques de atención y feed-forward.
    - Early stopping estricto.
    - Limitar número de capas del encoder.
    - Limitar dimensión del embedding.
    - Pooling simple o token final para salida escalar.

    **Regla práctica:** sin regularización fuerte, el modelo no generaliza.

#### **Resumen operativo**


- **Naive:** sin regularización.
- **Ridge / Lasso:** regularización integrada (L1 / L2).
- **MLP:** L2 + early stopping (+ dropout opcional).
- **GRU / LSTM:** early stopping + tamaño controlado + dropout.
- **TCN:** control de profundidad + dropout.
- **Transformer encoder:** dropout fuerte + límites estrictos.

La regularización no es un agregado opcional:  
define la capacidad efectiva del modelo y su comportamiento fuera de muestra.

La comparación final entre modelos se realiza recién en el **stage_07**.

## **5.2. Enfoque `seq2seq`**

### **5.2.1. Modelos propuestos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**
   - Predice “sin cambio” (por ejemplo, replica el último valor observado) en los 29 pasos.
   - Define el “piso” mínimo de performance.
   - Notebook: `stage_07_01_naive_model.ipynb`

2. **MLP (Direct Multi-step)**
   - Aplana $(29 \times 8)$ y predice $(29 \times 1)$.
   - Base neural simple y rápida para validar el pipeline.
   - Notebook: `stage_07_02_mlp_direct_multistep.ipynb`

#### **(B) Seq2Seq clásicos**


3. **Encoder–Decoder GRU**
    - Notebook: `stage_07_03_gru_seq2seq.ipynb`

4. **Encoder–Decoder LSTM**
    - Encoder resume la historia; decoder genera la secuencia futura.
    - Estables y comparables para intradía.
    - Notebook: `stage_07_04_lstm_seq2seq.ipynb`



#### **(C) Alternativa robusta no recurrente**


5. **TCN (Temporal Convolutional Network)**
   - Convoluciones causales dilatadas.
   - Buena relación performance/estabilidad.
   - Notebook: `stage_07_05_tcn_seq2seq.ipynb`

#### **(D) Modelo de atención**


6. **Transformer Seq2Seq**
   - Encoder–decoder con atención.
   - Requiere regularización y control cuidadoso, pero es candidato fuerte.
   - Notebook: `stage_07_06_transformer_seq2seq.ipynb`

#### **(E) Modelo avanzado con estructura temporal explícita**


7. **Temporal Fusion Transformer (TFT)**

- Modelo seq2seq con atención para series temporales multivariadas.
- Incorpora selección de variables y atención temporal para capturar dependencias pasadas y futuras.
- Alta capacidad para modelar patrones intradía complejos; referencia avanzada frente a LSTM, TCN y Transformer estándar.
- Notebook: `stage_07_07_tft_seq2seq.ipynb`

### **5.2.2. Regularización de modelos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    Regularización:
    No aplica.

    - No tiene parámetros entrenables.
    - No puede sobreajustar en sentido de Machine Learning.

    Rol:
    Define el piso mínimo de performance.
    Si un modelo entrenable no supera este baseline en validation, se descarta.

2. **MLP (Direct Multi-step)**

    Riesgo principal:
    Sobreajuste por capacidad del modelo.

    Regularización recomendada:

    - Penalización L2 (weight decay) como mecanismo principal.
    - Early stopping monitoreando la pérdida en validation.
    - Arquitectura limitada:
      - pocas capas (1–2),
      - número reducido de neuronas.
    - Dropout leve (opcional, solo si se observa sobreajuste claro).

    Interpretación según el libro:
    Modelo flexible que requiere regularización explícita para generalizar.

#### **(B) Seq2Seq clásicos**


3. **Encoder–Decoder GRU**

    Riesgo:
    Memorización de secuencias específicas del conjunto de entrenamiento.

    Regularización recomendada:

    - Early stopping como mecanismo principal.
    - Limitar la dimensión del hidden state.
    - Limitar el número de capas (1–2).
    - Dropout opcional entre capas, no dentro de la recurrencia.

    Nota:
    GRU es más estable que LSTM y suele requerir menor regularización.

4. **Encoder–Decoder LSTM**

    Riesgo:
    Alta capacidad combinada con memoria de largo plazo.

    Regularización recomendada:

    - Early stopping obligatorio.
    - Dropout en entradas y salidas (no recurrente).
    - Tamaño moderado del hidden state.
    - Número de capas limitado.
    - Penalización L2 suave en los pesos (opcional).

    Interpretación según el libro:
    Modelo potente que requiere regularización estructural y temporal.


#### **(C) Alternativa robusta no recurrente**


5. **TCN (Temporal Convolutional Network)**

    Riesgo:
    Campo receptivo excesivo o filtros redundantes.

    Regularización recomendada:

    - Limitar el número de filtros por capa.
    - Limitar la profundidad del modelo (dilataciones).
    - Dropout (especialmente efectivo en TCN).
    - Weight normalization si está disponible.
    - Early stopping.

    Ventaja clave:
    Suele generalizar mejor que modelos recurrentes con menor regularización agresiva.

#### **(D) Modelo de atención**


6. **Transformer Seq2Seq**

    Riesgo:
    Sobreajuste severo debido a alta capacidad.

    Regularización obligatoria:

    - Dropout alto en bloques de atención y feed-forward.
    - Early stopping estricto.
    - Limitar el número de capas.
    - Limitar la dimensión del embedding.
    - Label smoothing opcional en esquemas de pérdida multi-step.

    Regla práctica:
    Sin regularización fuerte, el modelo no generaliza.

#### **(E) Modelo avanzado con estructura temporal explícita**


7. **Temporal Fusion Transformer (TFT)**

    Riesgo:
    Alta capacidad, parcialmente mitigada por su diseño estructurado.

    Regularización integrada en la arquitectura:

    - Redes de selección de variables.
    - Mecanismos de gating.
    - Dropout configurable.
    - Early stopping.

    Aspectos que deben controlarse explícitamente:

    - Dimensión del hidden state.
    - Número de capas LSTM internas.
    - Nivel de dropout global.

    Interpretación según el libro:
    Modelo avanzado que regulariza principalmente por arquitectura, no solo por penalización.

#### **Resumen operativo**


- Naive: sin regularización.
- MLP: L2 + early stopping.
- GRU / LSTM: early stopping + tamaño controlado + dropout.
- TCN: control de profundidad + dropout.
- Transformer: dropout fuerte + límites estrictos.
- TFT: regularización arquitectónica + early stopping.

La regularización no es un agregado opcional.
Forma parte del diseño del modelo y determina su capacidad de generalización.

Cada modelo requiere un esquema de regularización distinto.
La comparación real entre ellos se realiza recién en el siguiente stage_07

## **5.3. Modelos según estructura de ventana**

### **5.3.1. Modelos con ventanas 2D (ventana aplanada)**

 Entrada:
  ```python
  (n_samples, window_size * n_features)
  ```
  No modela estructura temporal explícita.
  
  **Modelos:**
  - Naive (seq2one)
  - Ridge
  - Lasso
  - MLP (seq2one)
  - MLP Direct Multi-step (seq2seq)

### **5.3.2. Modelos con ventanas 3D (estructura temporal explícita)**

Entrada:
  
  ```python
  (n_samples, window_size, n_features)
  ```

Conserva la dimensión temporal y permite modelar dependencias dinámicas.

**Modelos:**
- Naive (seq2seq)
- LSTM (seq2one y seq2seq)
- GRU (seq2one y seq2seq)
- TCN (seq2one y seq2seq)
- Transformer encoder-only (seq2one)
- Transformer Seq2Seq
- Temporal Fusion Transformer (TFT)

# **6. Definición de la comparación Predicción vs Valor Real**

En este stage, la evaluación del modelo se realiza bajo los esquemas **`seq2one`** y **`seq2seq`**.  
Cada ventana histórica produce:

- una predicción escalar (seq2one), o  
- una secuencia futura completa (seq2seq).

La comparación siempre se realiza directamente entre predicción y valor real correspondiente, respetando la formulación original del problema.



## **6.1 Esquema de predicción**

### **6.1.1. Enfoque `seq2one``**

Para cada muestra $t$, el modelo recibe como entrada:

$$
X_t \in \mathbb{R}^{L \times N}
$$

correspondiente a $L$ minutos consecutivos con $N$ features por minuto.

El modelo produce una única predicción escalar:

$$
\hat{y}_t \in \mathbb{R}
$$

que representa el valor futuro del target definido (por ejemplo, $\Delta_h$ para un horizonte fijo $h$).

Se compara directamente contra el valor real observado:

$$
y_t \in \mathbb{R}
$$

La comparación es **escalar contra escalar**, sin:

- generar trayectorias,
- aplicar agregaciones intermedias,
- ni comparar secuencias completas.

Cada ventana histórica tiene una única predicción y un único valor real asociado.


### **6.1.2. Enfoque `seq2seq`**

Para cada muestra $t$, el modelo recibe como entrada:

$$
X_t \in \mathbb{R}^{L \times N}
$$

A partir de esta entrada, el modelo predice una secuencia futura completa:

$$
\hat{Y}_{t+1:t+L} \in \mathbb{R}^{L \times 1}
$$

es decir, un valor del target por cada uno de los $L$ pasos futuros.

La predicción se compara directamente contra la secuencia real futura observada:

$$
Y^{real}_{t+1:t+L} \in \mathbb{R}^{L \times 1}
$$

La comparación es **secuencia contra secuencia**, sin colapsar el target ni aplicar agregaciones previas.


## **6.2 Cálculo de métricas**

### **6.2.1. Enfoque `seq2one``**

A partir de la comparación $\hat{y}_t$ vs $y_t$, las métricas se calculan
sobre el conjunto completo de muestras del split correspondiente.

Métricas utilizadas:

- MAE
- RMSE
- R² (opcional)
- Métricas direccionales (signo de la predicción vs signo real)

Las métricas se agregan sobre todas las ventanas del split.

### **6.2.2. Enfoque `seq2seq`**


Las métricas se calculan bajo dos perspectivas:

**(A) Por paso temporal**

Comparación entre:

$$
\hat{y}_{t+k} \quad \text{vs} \quad y^{real}_{t+k}, \quad k = 1, \dots, L
$$

**(B) Sobre la trayectoria completa**

Error global entre:

$$
\hat{Y}_{t+1:t+L} \quad \text{vs} \quad Y^{real}_{t+1:t+L}
$$

No se realiza comparación contra un escalar ni contra un valor agregado final.

El problema se mantiene estrictamente bajo la formulación `seq2seq`.

## **6.3. Uso de los splits (criterio de evaluación)**


Siguiendo el criterio operativo del workflow:

- **TRAIN**: utilizado exclusivamente para el aprendizaje del modelo.
- **VALID**: utilizado para medir desempeño y descartar modelos que no generalizan.
- **TEST**: utilizado únicamente una vez finalizada la selección del modelo.

No se ajustan hiperparámetros en función del set de test.

El modelo se entrena solo con TRAIN, pero se evalúa con VALID para decidir si es candidato a pasar al stage siguiente.

En términos operativos:

- TRAIN → para aprender.
- VALID → para medir desempeño y comparar modelos.
- TEST → solo al final, una vez elegido el mejor modelo.

Este esquema evita contaminación de información y garantiza evaluación fuera de muestra.


## **6.4. Derivación de métricas**


A partir de la comparación entre predicción y valor real (según el esquema `seq2one` o `seq2seq`), se derivan dos grupos de métricas:

- **(A) Métricas de Machine Learning**

  - MAE
  - RMSE
  - R² (opcional)
  - Métricas direccionales (signo de la predicción vs signo real)

  Estas métricas se calculan sobre VALID durante el stage_06 para decidir qué modelos continúan.

- **(B) Métricas económicas**

  Una vez seleccionados los modelos candidatos, se derivan métricas económicas utilizando el delta real observado dentro de la ventana futura:

  - Valor esperado (EV)
  - Ratio TP/SL
  - Drawdown
  - Métricas de riesgo-retorno

  Estas métricas permiten traducir el desempeño estadístico en impacto operativo.

La comparación económica definitiva se realiza posteriormente en el stage_07.

# **7. Métricas de predicción (Machine Learning)**

La evaluación del desempeño se realiza exclusivamente fuera de muestra (VALID).  
El conjunto TEST se reserva únicamente para la evaluación final una vez seleccionado el modelo.

Las métricas se organizan según:

- Métricas principales (criterio de selección)
- Métricas complementarias (interpretación)
- Métricas diagnósticas (solo análisis interno)

## **7.1. Enfoque `seq2one`**

Dado que el problema consiste en la predicción de un valor escalar futuro, la comparación se realiza entre:

$$
\hat{y}_t \quad \text{vs} \quad y_t
$$

### **7.1.1 Métricas principales (criterio de selección)**

**1. MAE (Mean Absolute Error)**

Error absoluto medio entre predicción y valor real:

$$
\text{MAE} = \frac{1}{N} \sum_{t=1}^{N} |\hat{y}_t - y_t|
$$

**2. RMSE (Root Mean Squared Error)**

Raíz del error cuadrático medio:

$$
\text{RMSE} = \sqrt{\frac{1}{N} \sum_{t=1}^{N} (\hat{y}_t - y_t)^2}
$$

Estas métricas constituyen el **criterio principal de comparación y ranking** de modelos.



### **7.1.2 Métricas complementarias**


**3. Directional Accuracy (DA)**

$$
DA = P\left[\text{sign}(\hat{y}_t) = \text{sign}(y_t)\right]
$$

- No se utiliza como criterio principal de selección.
- Se reporta con fines interpretativos.
- Conecta la predicción con la dirección esperada del movimiento.

**4. Coeficiente de determinación ($R^2$)**

- Se reporta como métrica descriptiva.
- No se utiliza como criterio principal debido a su limitada estabilidad en series financieras.


## **7.2. Enfoque `seq2seq`**

Dado que el problema consiste en la predicción de una secuencia futura completa, la comparación se realiza entre:

$$
\hat{Y}_{t+1:t+L} \quad \text{vs} \quad Y_{t+1:t+L}
$$

### **7.2.1 Métricas principales (criterio de selección)**

Las métricas se calculan de forma global concatenando todos los pasos de la secuencia en un único vector.

**1. MAE global**

- Error absoluto medio sobre todos los pasos y todas las muestras.

**2. RMSE global**

- Raíz del error cuadrático medio sobre todos los pasos y todas las muestras.

Estas métricas constituyen el criterio principal de comparación y ranking de modelos.


### **7.2.2 Métricas diagnósticas**


**3. Error por horizonte**

$$
MAE(k), \quad k = 1, \dots, L
$$

Permite:

- Observar la degradación del error a medida que aumenta el horizonte.
- Analizar la estabilidad temporal del modelo.

Este análisis es estrictamente diagnóstico y no se utiliza para selección final.



### **7.2.3 Métricas complementarias**


**4. Directional Accuracy (DA_last)**

Coincidencia de signo en el último paso de la secuencia:

$$
k = L
$$

- Conecta con la dirección esperada al horizonte final.
- Se utiliza solo con fines interpretativos.

**5. Coeficiente de determinación ($R^2$)**

- Se reporta como métrica descriptiva.
- No se utiliza como criterio principal de comparación.



## **7.3 Jerarquía de métricas**

**Métricas principales (criterio de selección)**
- MAE
- RMSE

**Métricas complementarias**
- Directional Accuracy (DA o DA_last)
- R²

**Métricas diagnósticas (solo seq2seq)**
- MAE(k)

La selección y descarte de modelos se realiza exclusivamente en base a las métricas principales evaluadas sobre VALID.


## **7.4 Función de cálculo de métricas**

### **7.4.1. Función de cálculo de métricas seq2one**

In [1]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2one_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
    allow_seq_inputs_take_last: bool = False,
) -> dict:
    """
    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    """

    # 1) Convertir a np.ndarray y forzar float
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones hacia (n_samples,)
    def _to_1d(y: np.ndarray, name: str) -> np.ndarray:
        if y.ndim == 1:
            return y
        if y.ndim == 2:
            # (n, 1) -> (n,)
            if y.shape[1] == 1:
                return y.squeeze(1)
            # (n, seq_len) -> tomar último si se permite
            if allow_seq_inputs_take_last:
                return y[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} no es válido para seq2one. "
                f"Se esperaba (n_samples,) o (n_samples, 1)."
            )
        if y.ndim == 3 and y.shape[-1] == 1:
            # (n, seq_len, 1) -> (n, seq_len) y luego último si se permite
            y2 = y.squeeze(-1)
            if allow_seq_inputs_take_last:
                return y2[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} parece seq2seq. "
                f"Active allow_seq_inputs_take_last=True si quiere tomar el último paso."
            )
        raise ValueError(
            f"{name}.ndim={y.ndim} no es válido. "
            f"Se esperaba (n,), (n,1) o (n,seq_len) si allow_seq_inputs_take_last=True."
        )

    y_true = _to_1d(y_true, "y_true")
    y_pred = _to_1d(y_pred, "y_pred")

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true y y_pred deben tener el mismo shape. "
            f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    n_samples = int(y_true.shape[0])
    if n_samples == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica básica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred. "
                         "Limpie o enmascare antes de calcular métricas.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # 5) Métricas principales
    mae = float(abs_errors.mean())
    rmse = float(np.sqrt((errors ** 2).mean()))

    # 6) Métrica direccional (DA)
    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_n = int(mask.sum())
    else:
        da = float(np.mean(sign_true == sign_pred))
        da_n = n_samples

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "DA": da,
        "DA_n": da_n,  # cuántas muestras realmente aportaron a DA (si ignore_zeros=True)
    }

    # 7) R² opcional
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true, y_pred))

    return metrics

### **7.4.2. Función de cálculo de métricas seq2seq**

In [2]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2seq_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
) -> dict:
    """
    Calcula métricas comparables para modelos seq2seq.

    Retorna:
    - Métricas globales (MAE, RMSE)
    - Métricas del último paso (MAE_last, RMSE_last)
    - DA_last
    - MAE por paso (diagnóstico)
    """

    # 1) Convertir a np.ndarray
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones a (n_samples, seq_len)
    if y_true.ndim == 3 and y_true.shape[-1] == 1:
        y_true = y_true.squeeze(-1)
    if y_pred.ndim == 3 and y_pred.shape[-1] == 1:
        y_pred = y_pred.squeeze(-1)

    if y_true.ndim != 2 or y_pred.ndim != 2:
        raise ValueError(
            f"Se espera shape (n_samples, seq_len) o (n_samples, seq_len, 1). "
            f"Recibido y_true.ndim={y_true.ndim}, y_pred.ndim={y_pred.ndim}"
        )

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true y y_pred deben tener el mismo shape. "
            f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    n_samples, seq_len = y_true.shape
    if n_samples == 0 or seq_len == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # ===============================
    # MÉTRICAS GLOBALES (seq2seq puro)
    # ===============================
    mae = float(abs_errors.mean())
    rmse = float(np.sqrt((errors ** 2).mean()))

    # ===============================
    # MÉTRICAS ÚLTIMO PASO (comparables con seq2one)
    # ===============================
    y_true_last = y_true[:, -1]
    y_pred_last = y_pred[:, -1]

    errors_last = y_pred_last - y_true_last

    mae_last = float(np.abs(errors_last).mean())
    rmse_last = float(np.sqrt((errors_last ** 2).mean()))

    # ===============================
    # Directional Accuracy (último paso)
    # ===============================
    sign_true = np.sign(y_true_last)
    sign_pred = np.sign(y_pred_last)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da_last = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_last_n = int(mask.sum())
    else:
        da_last = float(np.mean(sign_true == sign_pred))
        da_last_n = int(n_samples)

    # ===============================
    # MAE por paso (diagnóstico)
    # ===============================
    mae_per_step = abs_errors.mean(axis=0)  # (seq_len,)

    metrics = {
        # Global seq2seq
        "MAE": mae,
        "RMSE": rmse,

        # Último paso (comparables con seq2one)
        "MAE_last": mae_last,
        "RMSE_last": rmse_last,

        # Dirección
        "DA_last": da_last,
        "DA_last_n": da_last_n,

        # Diagnóstico
        "MAE_per_step": mae_per_step.tolist(),
    }

    # ===============================
    # R² opcional
    # ===============================
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true.ravel(), y_pred.ravel()))
        metrics["R2_last"] = float(r2_score(y_true_last, y_pred_last))

    return metrics

In [4]:
#Ejemplo de uso:
#metrics = compute_seq2seq_metrics(y_true, y_pred)
#print(metrics["MAE"], metrics["RMSE"], metrics["DA_last"])

# **8. Métricas de SEQ2ONE**

In [22]:
import pandas as pd
from pathlib import Path
import os

In [23]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import pandas as pd
from pathlib import Path

def load_all_seq2one_metrics(
    *,
    models: list[str],
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    """
    Carga métricas seq2one para múltiples modelos y
    mantiene solo columnas estándar comparables.
    """

    cols = [
        "model", "split", "window_size", "target",
        "horizon_min", "MAE", "RMSE", "R2", "DA"
    ]

    dfs = []

    base_path = Path(base_dir)

    for name in models:
        path = base_path / f"seq2one_{name}_metrics.parquet"

        if not path.exists():
            continue

        df = pd.read_parquet(path)

        # Mantener solo columnas deseadas si existen
        keep_cols = [c for c in cols if c in df.columns]
        df = df[keep_cols].copy()

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=cols)

    df_all = pd.concat(dfs, ignore_index=True)

    # Orden consistente
    df_all = (
        df_all
        .sort_values(["model", "window_size", "target", "split"])
        .reset_index(drop=True)
    )

    return df_all


In [30]:
models_seq2one = ['naive', 'ridge', 'lasso', 'mlp', 'gru', 'lstm', 'tcn']

df_seq2one_all = load_all_seq2one_metrics(
    models=models_seq2one,
    base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics"
)

In [31]:
df_seq2one_all

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
0,gru,test,30,delta_60,60,53.044832,83.159082,0.144027,0.644839
1,gru,valid,30,delta_60,60,33.940821,48.203596,0.226267,0.663231
2,gru,test,30,delta_90,90,69.345335,104.527127,0.093254,0.619936
3,gru,valid,30,delta_90,90,43.885916,61.714770,0.157634,0.651104
4,gru,test,30,ret_60,60,0.003217,0.005895,-0.626744,0.590062
...,...,...,...,...,...,...,...,...,...
259,tcn,valid,180,delta_90,90,47.268811,65.357556,0.173360,0.617098
260,tcn,test,180,ret_60,60,0.005273,0.009266,-2.938613,0.527556
261,tcn,valid,180,ret_60,60,0.003007,0.004040,-0.558126,0.553760
262,tcn,test,180,ret_90,90,0.007677,0.011742,-3.349282,0.495239


In [33]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
DF = df_seq2one_all.copy()
PRIMARY_SPLIT = "test"
PRIMARY_METRIC = "R2"
SECONDARY_METRIC = "MAE"

# ============================================================
# CLEAN
# ============================================================
for c in ["window_size", "horizon_min", "MAE", "RMSE", "R2", "DA"]:
    if c in DF.columns:
        DF[c] = pd.to_numeric(DF[c], errors="coerce")

DF = DF.dropna(subset=["model","split","window_size","target","horizon_min","R2","MAE"]).copy()

DF["model"] = DF["model"].astype(str).str.lower()
DF["split"] = DF["split"].astype(str).str.lower()

# ============================================================
# SOLO TEST
# ============================================================
DFT = DF[DF["split"] == PRIMARY_SPLIT].copy()

print("Rows test:", len(DFT))
print("Models:", sorted(DFT["model"].unique()))
print("Targets:", sorted(DFT["target"].unique()))
print("Window sizes:", sorted(DFT["window_size"].unique()))
print("Horizons:", sorted(DFT["horizon_min"].unique()))

# ============================================================
# 1️⃣ MEJOR CONFIGURACIÓN GLOBAL
# ============================================================
best_global = (
    DFT.sort_values([PRIMARY_METRIC, SECONDARY_METRIC], ascending=[False, True])
       .head(1)
)

print("\n=== BEST OVERALL (TEST) ===")
display(best_global)

# ============================================================
# 2️⃣ MEJOR POR TARGET
# ============================================================
best_per_target = (
    DFT.sort_values([PRIMARY_METRIC, SECONDARY_METRIC], ascending=[False, True])
       .groupby("target", as_index=False)
       .head(1)
       .sort_values(PRIMARY_METRIC, ascending=False)
)

print("\n=== BEST PER TARGET (TEST) ===")
display(best_per_target[["target","horizon_min","model","window_size","R2","MAE","RMSE","DA"]])

# ============================================================
# 3️⃣ DELTA vs RETURN
# ============================================================
DFT["family"] = np.where(DFT["target"].str.startswith("delta"), "delta", "return")

family_summary = (
    DFT.groupby(["family","horizon_min"], as_index=False)
       .agg(
           R2_mean=("R2","mean"),
           R2_median=("R2","median"),
           MAE_mean=("MAE","mean"),
           DA_mean=("DA","mean"),
       )
       .sort_values(["family","horizon_min"])
)

print("\n=== DELTA vs RETURN (TEST) ===")
display(family_summary)

# ============================================================
# 4️⃣ MEJOR WINDOW SIZE (promedio R2)
# ============================================================
window_rank = (
    DFT.groupby("window_size", as_index=False)
       .agg(R2_mean=("R2","mean"),
            R2_median=("R2","median"))
       .sort_values("R2_mean", ascending=False)
)

print("\n=== WINDOW SIZE RANKING (TEST) ===")
display(window_rank)

# ============================================================
# 5️⃣ MEJOR MODELO GLOBAL
# ============================================================
model_rank = (
    DFT.groupby("model", as_index=False)
       .agg(R2_mean=("R2","mean"),
            R2_median=("R2","median"),
            MAE_mean=("MAE","mean"))
       .sort_values("R2_mean", ascending=False)
)

print("\n=== MODEL RANKING (TEST) ===")
display(model_rank)

# ============================================================
# 6️⃣ GANADOR POR (target, horizon, L)
# ============================================================
winner_per_cell = (
    DFT.sort_values(["target","horizon_min","window_size",PRIMARY_METRIC,SECONDARY_METRIC],
                    ascending=[True,True,True,False,True])
       .groupby(["target","horizon_min","window_size"], as_index=False)
       .head(1)
       .sort_values(PRIMARY_METRIC, ascending=False)
)

print("\n=== WINNER PER (target,horizon,L) ===")
display(winner_per_cell[["target","horizon_min","window_size","model","R2","MAE","DA"]].head(20))


Rows test: 112
Models: ['gru', 'lasso', 'lstm', 'mlp', 'ridge', 'tcn']
Targets: ['delta_60', 'delta_90', 'ret_60', 'ret_90']
Window sizes: [np.int64(30), np.int64(60), np.int64(90), np.int64(120), np.int64(180)]
Horizons: [np.int64(60), np.int64(90)]

=== BEST OVERALL (TEST) ===


,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
144,mlp,test,180,delta_60,60,38.11216,60.078973,0.570888,0.783965



=== BEST PER TARGET (TEST) ===


,target,horizon_min,model,window_size,R2,MAE,RMSE,DA
144,delta_60,60,mlp,180,0.570888,38.112160,60.078973,0.783965
146,delta_90,90,mlp,180,0.569329,49.753704,72.859485,0.789423
204,ret_60,60,ridge,60,0.299131,0.002097,0.003834,0.747543
206,ret_90,90,ridge,60,0.212914,0.002856,0.004922,0.723196



=== DELTA vs RETURN (TEST) ===


...

,window_size,R2_mean,R2_median
0,30,-1.490993,0.095348
1,60,-9.886982,0.148945
2,90,-24.713636,0.141237
3,120,-259.083266,0.153320
4,180,-1147.417155,-0.034049



=== MODEL RANKING (TEST) ===


,model,R2_mean,R2_median,MAE_mean
4,ridge,0.287400,0.264972,25.901361
1,lasso,0.129006,0.065117,29.023827
2,lstm,-0.046981,0.070243,30.408390
0,gru,-0.408411,-0.188832,30.183369
5,tcn,-0.486026,-0.073765,30.554230
3,mlp,-1271.535284,-6.109329,24.571954



=== WINNER PER (target,horizon,L) ===


,target,horizon_min,window_size,model,R2,MAE,DA
144,delta_60,60,180,mlp,0.570888,38.112160,0.783965
146,delta_90,90,180,mlp,0.569329,49.753704,0.789423
136,delta_60,60,120,mlp,0.518663,39.748844,0.741398
120,delta_60,60,60,mlp,0.496504,38.923520,0.738393
128,delta_60,60,90,mlp,0.481965,40.382358,0.729042
138,delta_90,90,120,mlp,0.414109,56.524240,0.703310
130,delta_90,90,90,mlp,0.379493,56.944418,0.689536
122,delta_90,90,60,mlp,0.367589,56.306911,0.699427
204,ret_60,60,60,ridge,0.299131,0.002097,0.747543
212,ret_60,60,90,ridge,0.275536,0.002164,0.733276


🎯 Qué te va a responder esto

1. 🏆 El mejor modelo absoluto (test)
2. 🥇 El mejor modelo por target
3. 🔥 Si delta realmente supera a return (estadísticamente)
4. 📏 Cuál window_size es mejor en promedio
5. 🤖 Qué arquitectura gana en promedio
6. 📊 El ganador por cada combinación (target, horizonte, ventana)